# Part 3: Prompt Engineering & Evaluation

In [1]:
# --- Reconnect to Foundry (run this first after opening a new notebook) ---
import subprocess, sys, json, shutil
from pathlib import Path

def run_cmd(args, check=True):
    exe = shutil.which(args[0])
    if exe: args = [exe] + args[1:]
    result = subprocess.run(args, capture_output=True, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(result.stderr.strip())
    return result

SUFFIX = run_cmd(["az", "ad", "signed-in-user", "show", "--query", "id", "-o", "tsv"]).stdout.strip()[:6]
RESOURCE_GROUP = f"rg-foundry-workshop-{SUFFIX}"

outputs = json.loads(run_cmd([
    "az", "deployment", "group", "show", "-g", RESOURCE_GROUP, "-n", "main",
    "--query", "properties.outputs", "-o", "json"
]).stdout)

PROJECT_ENDPOINT = outputs["projectEndpoint"]["value"]
MODEL_DEPLOYMENT_NAME = outputs["modelDeploymentName"]["value"]
APP_INSIGHTS_CONN_STR = outputs["appInsightsConnectionString"]["value"]

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, FileSearchTool, WebSearchPreviewTool, FunctionTool
from azure.identity import DefaultAzureCredential, AzureCliCredential

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
openai_client = project_client.get_openai_client()
print(f"✅ Connected to: {PROJECT_ENDPOINT}")
print(f"   Model: {MODEL_DEPLOYMENT_NAME}")

✅ Connected to: https://fndry-ws-c9db47.services.ai.azure.com/api/projects/workshop-project
   Model: gpt-4.1


---
## Section 3: Prompt Engineering (~7 min)

The quality of an agent depends heavily on its **instructions** (system prompt).
Key patterns:
- **Persona**: Define who the agent is and its expertise
- **Constraints**: What it should and shouldn't do
- **Output format**: Specify structure (JSON, markdown, etc.)
- **Few-shot examples**: Show desired input/output pairs

In [2]:
# --- 3.1 Basic vs. Engineered Prompt ---
# Compare a vague prompt with a well-engineered one.

QUESTION = "Analyze the potential of AI in quality control for manufacturing."

# Basic prompt
basic_agent = project_client.agents.create_version(
    agent_name="basic-analyst",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions="You are helpful.",
    ),
)
r_basic = openai_client.responses.create(
    input=QUESTION,
    extra_body={"agent_reference": {"name": basic_agent.name, "type": "agent_reference"}},
)

# Engineered prompt
eng_agent = project_client.agents.create_version(
    agent_name="engineered-analyst",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a senior industrial technology analyst with 15 years of experience "
            "in manufacturing and quality assurance.\n\n"
            "RULES:\n"
            "- Structure your analysis with clear headings\n"
            "- Include specific, quantifiable benefits where possible\n"
            "- Address both opportunities AND risks\n"
            "- Keep responses under 300 words\n"
            "- End with 3 actionable recommendations\n\n"
            "FORMAT: Use markdown with ## headings."
        ),
    ),
)
r_eng = openai_client.responses.create(
    input=QUESTION,
    extra_body={"agent_reference": {"name": eng_agent.name, "type": "agent_reference"}},
)

print("=" * 60)
print("BASIC PROMPT RESPONSE:")
print("=" * 60)
print(r_basic.output_text[:500])
print(f"\n... ({len(r_basic.output_text)} chars total)")
print("\n" + "=" * 60)
print("ENGINEERED PROMPT RESPONSE:")
print("=" * 60)
print(r_eng.output_text[:800])
print(f"\n... ({len(r_eng.output_text)} chars total)")


BASIC PROMPT RESPONSE:
AI holds significant potential to revolutionize quality control (QC) in manufacturing by increasing accuracy, efficiency, and scalability. Here's an analysis of its implications, benefits, challenges, and future potential:

### 1. **Enhanced Defect Detection**
- **Computer Vision:** AI-driven computer vision systems can inspect products in real time to identify defects like cracks, misalignments, or surface imperfections, often beyond the precision and speed of human inspectors.
- **Pattern Reco

... (3548 chars total)

ENGINEERED PROMPT RESPONSE:
## Overview

AI is rapidly transforming quality control (QC) in manufacturing by automating defect detection, reducing human error, and providing actionable insights from production data. Adoption is rising due to advancements in machine vision, predictive analytics, and IoT integration.

---

## Opportunities

### 1. Enhanced Defect Detection
- **Accuracy**: Machine vision AI can detect defects with >99% accuracy, outp

In [3]:
# --- 3.2 Structured JSON Output ---
# Instruct the agent to return valid JSON with a predefined schema.

json_agent = project_client.agents.create_version(
    agent_name="structured-analyst",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a business analyst. Always respond with ONLY valid JSON "
            "(no markdown fences, no extra text) using this schema:\n"
            '{\n'
            '  "executive_summary": "2-3 sentence overview",\n'
            '  "key_findings": ["finding 1", "finding 2", "finding 3"],\n'
            '  "recommendations": ["rec 1", "rec 2"],\n'
            '  "risk_level": "low | medium | high"\n'
            '}'
        ),
    ),
)
r_json = openai_client.responses.create(
    input="Assess the business case for deploying AI-powered visual inspection in a food processing plant.",
    extra_body={"agent_reference": {"name": json_agent.name, "type": "agent_reference"}},
)

print("Raw response:")
print(r_json.output_text)

# Parse and pretty-print
try:
    parsed = json.loads(r_json.output_text)
    print("\n✅ Valid JSON! Parsed fields:")
    for key, value in parsed.items():
        print(f"  {key}: {value}")
except json.JSONDecodeError as e:
    print(f"\n⚠️ JSON parse error: {e}")


Raw response:
{
  "executive_summary": "Deploying AI-powered visual inspection in a food processing plant offers the potential to enhance product quality, reduce labor costs, and minimize human error. The investment aligns with industry trends toward automation and data-driven quality assurance, positioning the plant competitively for future growth.",
  "key_findings": [
    "AI-based visual inspection can improve defect detection accuracy and consistency compared to manual methods.",
    "Initial capital investment is required, but operational savings from reduced waste and labor can yield a significant ROI over time.",
    "Integrating AI systems may necessitate workforce reskilling and robust change management strategies."
  ],
  "recommendations": [
    "Conduct a pilot implementation in a single production line to validate business benefits before scaling plant-wide.",
    "Develop a comprehensive training and transition plan to support workforce adaptation and maintain operationa

In [4]:
# --- 3.3 Few-Shot Prompting ---
# Include example Q&A pairs in the instructions to guide the agent's style.

fewshot_agent = project_client.agents.create_version(
    agent_name="fewshot-advisor",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a concise technical advisor. Follow the style shown in these examples:\n\n"
            "USER: Should we migrate to microservices?\n"
            "ASSISTANT: **Verdict**: Only if your team exceeds 20 engineers.\n"
            "**Why**: Microservices add operational overhead (CI/CD, observability, networking). "
            "Below 20 engineers, a modular monolith delivers the same benefits with less complexity.\n"
            "**Action**: Start with a modular monolith; extract services only when team boundaries demand it.\n\n"
            "USER: Is Kubernetes necessary for our 3-service app?\n"
            "ASSISTANT: **Verdict**: No — use Azure Container Apps instead.\n"
            "**Why**: K8s is overkill for <10 services. Container Apps provides scale-to-zero, "
            "managed ingress, and Dapr integration without cluster management.\n"
            "**Action**: Deploy to Container Apps; re-evaluate K8s at 10+ services.\n\n"
            "Always use this exact format: **Verdict**, **Why**, **Action**."
        ),
    ),
)

r_fs = openai_client.responses.create(
    input="Should we build our own LLM evaluation framework or use an existing one?",
    extra_body={"agent_reference": {"name": fewshot_agent.name, "type": "agent_reference"}},
)
print("🧑 Should we build our own LLM evaluation framework or use an existing one?\n")
print(f"🤖 {r_fs.output_text}")


🧑 Should we build our own LLM evaluation framework or use an existing one?

🤖 **Verdict**: Use an existing LLM evaluation framework.

**Why**: Mature frameworks (like HELM, LM-Eval, or Ragna) provide standard metrics, benchmarks, extensibility, and integration with common models—removing the need to reinvent reliability, reproducibility, and reporting. Custom frameworks divert resources and typically underperform on edge cases without significant investment.

**Action**: Evaluate your requirements against popular open-source solutions; extend via plugins if needed. Build in-house only if you have niche, unsupported evaluation needs.


---
## Section 4: Multi-Step Reasoning & Evaluation (~10 min)

In this section we'll:
1. Build an agent that performs **multi-step reasoning** (analyze → research → synthesize)
2. **Evaluate** agent quality using Microsoft's built-in evaluators

In [5]:
# --- 4.1 Multi-Step Reasoning Agent ---
# This agent analyzes a vague request, researches feasibility, and produces
# a structured requirements document — all in one turn via chain-of-thought prompting.

requirements_agent = project_client.agents.create_version(
    agent_name="requirements-analyst",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a senior business analyst. When given a vague request, follow these steps:\n\n"
            "STEP 1 — ANALYSIS: Identify what's being asked. List assumptions and ambiguities.\n"
            "STEP 2 — CLARIFYING QUESTIONS: List 3-5 questions you would ask the stakeholder.\n"
            "STEP 3 — RESEARCH: Based on your knowledge, assess technical feasibility.\n"
            "STEP 4 — REQUIREMENTS DOCUMENT: Produce a structured output with:\n"
            "  - Objective (1-2 sentences)\n"
            "  - Functional Requirements (numbered list)\n"
            "  - Non-Functional Requirements (numbered list)\n"
            "  - Risks & Mitigations (table format)\n"
            "  - Estimated Complexity: Low | Medium | High\n\n"
            "Show your reasoning at each step. Use markdown formatting."
        ),
        tools=[WebSearchPreviewTool()],
    ),
)

multi_step_query = "We need an AI system that can read our equipment manuals and answer technician questions."

r_ms = openai_client.responses.create(
    input=multi_step_query,
    extra_body={"agent_reference": {"name": requirements_agent.name, "type": "agent_reference"}},
)
print(f"🧑 {multi_step_query}\n")
print(f"🤖 {r_ms.output_text}")

# Save for evaluation
multi_step_response = r_ms.output_text


🧑 We need an AI system that can read our equipment manuals and answer technician questions.

🤖 Certainly! Let’s proceed step by step.

---

## STEP 1 — ANALYSIS

**What’s being asked:**  
You're requesting an AI-driven system capable of ingesting (reading) equipment manuals and providing answers to technician queries about those manuals.

**Assumptions & Ambiguities:**  
- *Type of equipment*: Unspecified.
- *Format of manuals*: Could be PDFs, Word docs, web pages, or paper.
- *Types of questions*: Are they troubleshooting, operational, safety, etc.?
- *Integration*: Is this a standalone tool, or should it integrate into an existing platform?
- *Security & Access*: Who can access this information?

---

## STEP 2 — CLARIFYING QUESTIONS

1. **What types and formats are the equipment manuals available in (PDF, DOCX, HTML, etc.)?**
2. **What are the most common types of questions technicians will ask (troubleshooting, procedures, technical specs, etc.)?**
3. **Do you want this system to i

In [6]:
# --- 4.2 Single-Response Evaluation ---
# Evaluate the agent's output using built-in quality evaluators.
# These use a "judge" LLM to score responses on various dimensions.

from azure.ai.evaluation import (
    RelevanceEvaluator,
    CoherenceEvaluator,
    FluencyEvaluator,
)

# The evaluators need a model config — we reuse the same deployment
# via the project endpoint to authenticate through DefaultAzureCredential.
eval_model_config = {
    "azure_endpoint": PROJECT_ENDPOINT.rsplit("/api/projects", 1)[0],
    "azure_deployment": MODEL_DEPLOYMENT_NAME,
    "api_version": "2025-04-01-preview",
}

relevance = RelevanceEvaluator(eval_model_config)
coherence = CoherenceEvaluator(eval_model_config)
fluency   = FluencyEvaluator(eval_model_config)

print("Evaluating the multi-step reasoning response...\n")

for name, evaluator in [("Relevance", relevance), ("Coherence", coherence), ("Fluency", fluency)]:
    result = evaluator(
        query=multi_step_query,
        response=multi_step_response,
    )
    score_key = [k for k in result if not k.endswith("_reason") and not k.endswith("_result") and not k.endswith("_threshold")]
    score = result.get(score_key[0], "N/A") if score_key else "N/A"
    reason = result.get(f"{score_key[0]}_reason", "") if score_key else ""
    print(f"  {name}: {score}/5")
    if reason:
        print(f"    Reason: {reason[:120]}...")
    print()

Evaluating the multi-step reasoning response...

  Relevance: 5.0/5
    Reason: The response thoroughly analyzes the user's request, asks clarifying questions, explains technical feasibility, and prov...

  Coherence: 5.0/5
    Reason: The response is exceptionally coherent, with a logical, seamless structure, clear transitions, and thorough coverage of ...

  Fluency: 5.0/5
    Reason: The response demonstrates proficient to exceptional fluency, with sophisticated structure, vocabulary, and clarity, maki...



In [7]:
# --- 4.3 Batch Evaluation with Synthetic Dataset ---
# Run evaluation across multiple pre-built query/response pairs.
# Results are published to the Foundry portal.

from azure.ai.evaluation import evaluate, GroundednessEvaluator

eval_data_path = str(Path.cwd() / "sample_data" / "eval_dataset.jsonl")

groundedness = GroundednessEvaluator(eval_model_config)

print("⏳ Running batch evaluation (this may take 1-2 min)...\n")

result = evaluate(
    data=eval_data_path,
    evaluators={
        "relevance": relevance,
        "coherence": coherence,
        "fluency": fluency,
        "groundedness": groundedness,
    },
    evaluator_config={
        "groundedness": {
            "column_mapping": {
                "query": "${data.query}",
                "context": "${data.context}",
                "response": "${data.response}",
            }
        }
    },
    azure_ai_project=PROJECT_ENDPOINT,
    output_path=str(Path.cwd() / "eval_results.json"),
)

print("=== Aggregate Metrics ===")
for metric, value in result.get("metrics", {}).items():
    print(f"  {metric}: {value}")

studio_url = result.get("studio_url", "")
if studio_url:
    print(f"\n🔗 View in Foundry Portal: {studio_url}")
print("\n✅ Batch evaluation complete!")


⏳ Running batch evaluation (this may take 1-2 min)...

2026-04-16 10:43:53 +0200 6270464000 execution.bulk     INFO     Finished 1 / 6 lines.
2026-04-16 10:43:53 +0200 6270464000 execution.bulk     INFO     Average execution time for completed lines: 3.48 seconds. Estimated time for incomplete lines: 17.4 seconds.
2026-04-16 10:43:54 +0200 6270464000 execution.bulk     INFO     Finished 2 / 6 lines.
2026-04-16 10:43:54 +0200 6270464000 execution.bulk     INFO     Average execution time for completed lines: 2.01 seconds. Estimated time for incomplete lines: 8.04 seconds.
2026-04-16 10:43:54 +0200 6270464000 execution.bulk     INFO     Finished 3 / 6 lines.
2026-04-16 10:43:54 +0200 6270464000 execution.bulk     INFO     Average execution time for completed lines: 1.39 seconds. Estimated time for incomplete lines: 4.17 seconds.
2026-04-16 10:43:54 +0200 6270464000 execution.bulk     INFO     Finished 4 / 6 lines.
2026-04-16 10:43:54 +0200 6270464000 execution.bulk     INFO     Average ex

Aggregated metrics for evaluator is not a dictionary will not be logged as metrics


======= Run Summary =======

Run name: "relevance_20260416_084350_210233"
Run status: "Completed"
Start time: "2026-04-16 08:43:50.210233+00:00"
Duration: "0:00:05.233517"

2026-04-16 10:43:55 +0200 6304116736 execution.bulk     INFO     Finished 3 / 6 lines.
2026-04-16 10:43:55 +0200 6304116736 execution.bulk     INFO     Average execution time for completed lines: 1.75 seconds. Estimated time for incomplete lines: 5.25 seconds.
2026-04-16 10:43:55 +0200 6320943104 execution.bulk     INFO     Finished 1 / 6 lines.
2026-04-16 10:43:55 +0200 6320943104 execution.bulk     INFO     Average execution time for completed lines: 5.25 seconds. Estimated time for incomplete lines: 26.25 seconds.
2026-04-16 10:43:55 +0200 6304116736 execution.bulk     INFO     Finished 4 / 6 lines.
2026-04-16 10:43:55 +0200 6304116736 execution.bulk     INFO     Average execution time for completed lines: 1.33 seconds. Estimated time for incomplete lines: 2.66 seconds.
2026-04-16 10:43:55 +0200 6320943104 execut

Aggregated metrics for evaluator is not a dictionary will not be logged as metrics
Aggregated metrics for evaluator is not a dictionary will not be logged as metrics
Aggregated metrics for evaluator is not a dictionary will not be logged as metrics


======= Run Summary =======

Run name: "fluency_20260416_084350_210569"
Run status: "Completed"
Start time: "2026-04-16 08:43:50.210569+00:00"
Duration: "0:00:07.234628"

======= Run Summary =======

Run name: "coherence_20260416_084350_209937"
Run status: "Completed"
Start time: "2026-04-16 08:43:50.209937+00:00"
Duration: "0:00:07.236117"

======= Combined Run Summary (Per Evaluator) =======

{
    "relevance": {
        "status": "Completed",
        "duration": "0:00:05.233517",
        "completed_lines": 6,
        "failed_lines": 0,
        "log_path": null,
        "per_line_errors": {},
        "error_message": null,
        "error_code": null
    },
    "coherence": {
        "status": "Completed",
        "duration": "0:00:07.236117",
        "completed_lines": 6,
        "failed_lines": 0,
        "log_path": null,
        "per_line_errors": {},
        "error_message": null,
        "error_code": null
    },
    "fluency": {
        "status": "Completed",
        "duration"

> 💡 **View results**: Click the link above to see per-row scores and aggregate metrics.
> Note: The `azure-ai-evaluation` SDK currently publishes results to the **classic Foundry portal**
> (under Evaluation), not the new Foundry portal. This is a known SDK limitation.


---
Proceed to `04-orchestration.ipynb`.